In [ ]:
# Faith Yang & Robert Armenta (Monday/Wednesday Lab)
# Linear Model
# Dataset: https://www.kaggle.com/datasets/kulkarniparth09/california-house-details

In [ ]:
# Import libraries
import pandas as pd
import kagglehub
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

In [ ]:
# Download dataset
path = kagglehub.dataset_download("kulkarniparth09/california-house-details")

# Load dataset (update filename if needed)
df = pd.read_csv("/content/housing.csv")

100%|██████████| 474k/474k [00:00<00:00, 578kB/s]

Extracting files...


In [ ]:
# Data cleaning
# Check missing values
print("Missing values:")
print(df.isnull().sum())
print()

Missing values:
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64



In [ ]:
# Drop missing values
df = df.dropna()

print("Missing values removed.")
print()

Missing values removed.



In [ ]:
# Encode categorical variables
label_encoders = {}

for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

print("Categorical variables encoded.")
print()

Categorical variables encoded.



In [ ]:
# dataset structure overiew including data types and missing values
df.info()
print(df.shape)

# number of unique values in each columnn
df.nunique()

# display all column names in the dataset
print(df.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
Index: 20433 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20433 non-null  float64
 1   latitude            20433 non-null  float64
 2   housing_median_age  20433 non-null  float64
 3   total_rooms         20433 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20433 non-null  float64
 6   households          20433 non-null  float64
 7   median_income       20433 non-null  float64
 8   median_house_value  20433 non-null  float64
 9   ocean_proximity     20433 non-null  int64  
dtypes: float64(9), int64(1)
memory usage: 1.7 MB
(20433, 10)
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']


In [ ]:
# Create a binary classification target:
# High price (1) vs Low price (0)
median_price = df['median_house_value'].median()
df['price_category'] = (df['median_house_value'] > median_price).astype(int)

In [ ]:
# Select features (adjust based on dataset columns)
X = df[['total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']].copy()
y = df['price_category']

# Impute missing values in X with the median of each column
# This is necessary because SVC does not handle NaNs
X.fillna(X.median(), inplace=True)

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Scale data (important for SVM)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# SVM Linear Model
svm_model = SVC(kernel='linear', C=1.0)
svm_model.fit(X_train, y_train)

SVC(kernel='linear')

In [ ]:
# Predictions
y_pred = svm_model.predict(X_test)

In [ ]:
# Project Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 79.13%

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.80      0.79      2065
           1       0.79      0.78      0.79      2022

    accuracy                           0.79      4087
   macro avg       0.79      0.79      0.79      4087
weighted avg       0.79      0.79      0.79      4087


Confusion Matrix:
[[1652  413]
 [ 440 1582]]


In [ ]:
# Support vectors
print(f"\nNumber of Support Vectors: {len(svm_model.support_vectors_)}")


Number of Support Vectors: 8396


In [ ]:
# New Data Entry (Prediction)
# Example: new house data - should have 5 features matching the model's input
# Features used: total_rooms, total_bedrooms, population, households, median_income
new_house = [[2500, 450, 1200, 400, 4.5]] # Example values for the 5 features
new_house_scaled = scaler.transform(new_house)

prediction = svm_model.predict(new_house_scaled)

print("\nNew Data Prediction:")
if prediction[0] == 1:
    print("This house is predicted to be HIGH price")
else:
    print("This house is predicted to be LOW price")


New Data Prediction:
This house is predicted to be HIGH price


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


This project uses a Support Vector Machine (SVM) with a linear kernel to classify houses as either high-priced or low-priced. The model learns by drawing a boundary (line) that best separates the two groups based on features like area, bedrooms, bathrooms, and stories. After training the model, we tested it using unseen data and measured its performance using accuracy, confusion matrix, and classification report. The results show how well the model predicts house prices and whether it makes errors. We also tested the model with a new house example to see how it predicts in real-world situations, showing that the model can be used for future decision-making.